# Intro

## Global parameters

In [2]:
MAX_TIME_HOURS=1
ALPHA=0.01
_PVAL_FLOOR=10**-6

## Modules

### Standard

In [3]:
import os, pickle, platform, sys
import numpy as np
import copy
import torch

In [4]:
from collections import defaultdict

In [5]:
import dcms
from dcms.models import DCMModel, DECMModel, qDECMModel, DWCMModel

In [6]:
import matplotlib.pyplot as plt
plt.rcParams['axes.linewidth'] = 2
plt.rcParams['xtick.major.size'] = 10
plt.rcParams['xtick.major.width'] = 2
plt.rcParams['ytick.major.size'] = 10
plt.rcParams['ytick.major.width'] = 2

plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14

plt.rcParams['xtick.minor.size'] = 5
plt.rcParams['xtick.minor.width'] = 1
plt.rcParams['ytick.minor.size'] = 5
plt.rcParams['ytick.minor.width'] = 1
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter

In [7]:
from scipy.stats import spearmanr

In [8]:
from tqdm.notebook import tqdm, trange

In [9]:
import datetime as dt

In [10]:
from bowtie import edges2bowtie

In [11]:
from tabulate import tabulate

### Home made

In [12]:
if platform.system() == 'Darwin':
    print('Air!')
    HOME = '/Users/fabio/Documents/Lavoro/PythonFiles/bowtie2_py310/bowtie2/'
elif platform.system() == 'Linux':
    print('Stella!')
    HOME = '/home/sarawalk/bowtie2_py39/bowtie2/'
else:
    raise RuntimeError(f"Unsupported OS: {platform.system()}")

sys.path.insert(0, HOME)

Air!


In [13]:
from auxiliary_functions import el2ks, bic

In [14]:
from sam_bowtie import block_and_fluxes as bnf

In [15]:
from bowtie_plot_functions import plot_bowtie_blocks, plot_bowtie_fluxes, _add_colorbar
from bowtie_plot_functions import _fdr as fdr

## Load data

In [16]:
DATA_FOLDER=HOME+'dati_elezioni/'
TEST_FOLDER=HOME+'tests/'
PVALUE_FOLDER=HOME+'pvalues/'
GUARINO_FOLDER=HOME+'guarino_files/'
BIPARTITE_FOLDER=HOME+'BiDCM/'
PLOT_FOLDER=HOME+'plots/'

## Names for $\LaTeX$
#### Names as in the draft

In [33]:
def parse_dico_names(filepath):
    with open(filepath, 'r') as f:
        lines = [line.strip().split() for line in f]
        # such a command creates a list
        # in which each element is a list of tokens of a line in the file
        # remarkably, there is an empty element
        # before each dataset name
        

    result = {}
    current_key = None

    for tokens in lines:
        if not tokens:
            current_key = None
        elif current_key is None:
            # prima riga non vuota del gruppo: nome del dataset
            current_key = tokens[0]
            result[current_key] = {}
        else:
            # riga tipo ['5:', 'journalists', '&', 'Media']
            idx = int(tokens[0].rstrip(':'))
            label = ' '.join(tokens[1:])
            result[current_key][idx] = label

    return result

In [34]:
cacca=parse_dico_names(GUARINO_FOLDER+'all_dico_labels.txt')

In [35]:
cacca['quirinale']

{5: 'journalists & Media',
 2: 'M5S',
 0: 'journalists & IV & Azione & +Europa & Media',
 4: 'Lega & FDI',
 1: 'PD',
 3: 'Media & journalists',
 6: 'FI'}

In [36]:
name_changer={'c0': 'Crisis - Centrists', 'c1': 'Crisis - Centre-right', 'c2': 'Crisis - M5S', 'c3': 'Crisis - PD', 'c4': 'Crisis - Media', 'e0': 'Elections - Centre-right', 'e1': 'Elections - Centre-left', 'e2': 'Elections - M5S', 'e3': 'Elections - Centrists', 'e4': 'Elections - Others', 'e5': 'Elections - Media', 'e6': 'Elections - Others 2', 'p0': 'President -  Centrists', 'p1': 'President -  PD', 'p2': 'President -  M5S', 'p3': 'President -  Media', 'p4': 'President -  Right', 'p5': 'President -  Others', 'p6': 'President -  FI'}

#### Proper names for the Latex table

In [37]:
def sci_latex(x, dec=2, bold=False):
    # da Claude
    mant, exp = f'{x:.{dec}e}'.split('e')
    if bold:
        return f'$\\mathbf{{{mant}\\times 10^{{{int(exp)}}}}}$'
    return f'${mant}\\times 10^{{{int(exp)}}}$'


# Where are we?

## Check the files

In [17]:
os.listdir(TEST_FOLDER)

['quirinale_dico0_dwcm_final.pkl',
 'crisi_dico3_qdecm.pkl',
 'batch2608_ita_elections_dico6_decm.pkl',
 'decm_validation',
 'ita_elections_dico3_dwcm_final.pkl',
 'crisi_run9.log',
 'crisi_dico0_dwcm_final.pkl',
 'crisi_dico4_decm.pkl',
 'ita_elections_dico1_decm_conv.pkl',
 'ita_elections_dico4_dwcm_final.pkl',
 'crisis_qdecm_new_theta_nprocs_8.pkl',
 'batch2608_quirinale_dico4_decm.pkl',
 'ita_elections_dico5_dwcm_final.pkl',
 'crisi_dico1_dwcm_final.pkl',
 'quirinale_dico6_dwcm_final.pkl',
 'ita_elections_dico2_dwcm_final.pkl',
 'quirinale_dico1_qdecm.pkl',
 'quirinale_dico1_dwcm_final.pkl',
 'crisi_dico2_decm_and_10_gamma_0.0_hub_0_gauge_min.pkl',
 'ita_elections_dico1_qdecm.pkl',
 'ita_elections_dico0_dwcm_final.pkl',
 'batch2608_quirinale_dico2_decm.pkl',
 'batch2608_quirinale_dico3_decm.pkl',
 'ita_elections_dico3_decm_stella.pkl',
 'quirinale_dico4_decm_conv.pkl',
 'crisi_dico4_dwcm_final.pkl',
 'quirinale_dico3_dwcm_final.pkl',
 'crisi_dico2_qdecm.pkl',
 'crisi_run_symgs.log'

## Get all converged solutions

In [18]:
solution_files=[f for f in os.listdir(TEST_FOLDER) if f.endswith('.pkl')]
solution_files.sort()

In [19]:
final_dict={}
for solution in solution_files:
    # get the model type from the solution filename
    if 'qdecm' in solution.lower():
        model_type='qDECM'
    elif 'decm' in solution.lower():
        model_type='DECM'
    elif 'dwcm' in solution.lower():
        model_type='DWCM'
    else:
        print(f"Unknown model type in solution {solution}. Skipping.")
        continue

    # get the dataset from the solution filename
    if 'elections' in solution.lower():
        name='e'
    elif 'crisi' in solution.lower():
        name='c'
    elif 'quirinale' in solution.lower():
        name='p'
    else:
        print(f"Unknown dataset in solution {solution}. Skipping.")
        continue
    
    try:
        i_dico=solution.index('dico')
        dico=solution[i_dico+4]
    except ValueError:
        print(f"No 'dico' found in solution {solution}. Skipping.")
        continue

    dataset=f"{name}{dico}"
    if os.path.exists(TEST_FOLDER+solution):
        try:
            with open(TEST_FOLDER+solution, 'rb') as f:
                _temp = pickle.load(f)
        except Exception as e:
            print(f"{solution}: Error occurred while loading the solution: {e}. Deleting the file and skipping.")
            os.remove(TEST_FOLDER+solution)
            continue
        try:
            if _temp.sol.mre < 1e-5:  # Check if the model converged
                print(f"{solution}: Model converged.")
                if dataset not in final_dict:
                    final_dict[dataset]={}
                if model_type not in final_dict[dataset]:
                    final_dict[dataset][model_type]=_temp
                else:
                    print(f"{solution}: Converging model already exists in final_dict. Skipping.")
            else:
                print(f"{solution}: Model did not converge.")
        except AttributeError:
            print(f"{solution}: No 'sol' attribute in the loaded data.")

batch2608_crisi_dico1_decm.pkl: Model did not converge.
batch2608_crisi_dico3_decm.pkl: Model converged.
batch2608_crisi_dico4_decm.pkl: Model converged.
batch2608_ita_elections_dico0_decm.pkl: Model did not converge.
batch2608_ita_elections_dico1_decm.pkl: Model did not converge.
batch2608_ita_elections_dico4_decm.pkl: Model converged.
batch2608_ita_elections_dico5_decm.pkl: Model converged.
batch2608_ita_elections_dico6_decm.pkl: Model converged.
batch2608_quirinale_dico0_decm.pkl: Model converged.
batch2608_quirinale_dico1_decm.pkl: Model converged.
batch2608_quirinale_dico2_decm.pkl: Model converged.
batch2608_quirinale_dico3_decm.pkl: Model converged.
batch2608_quirinale_dico4_decm.pkl: Model did not converge.
crisi_dico0_decm_conv.pkl: Model converged.
crisi_dico0_dwcm_final.pkl: Model converged.
crisi_dico0_qdecm.pkl: Model converged.
crisi_dico1_decm_conv.pkl: Model converged.
crisi_dico1_dwcm_final.pkl: Model converged.
crisi_dico1_qdecm.pkl: Model converged.
crisi_dico2_decm.

### Check

In [ ]:
_keys=list(final_dict.keys())
_keys.sort()

In [ ]:
_three_of_us=[key for key in _keys if len(final_dict[key]) == 3]
_three_of_us

['c0',
 'c1',
 'c2',
 'c3',
 'c4',
 'e0',
 'e1',
 'e2',
 'e3',
 'e4',
 'e5',
 'e6',
 'p0',
 'p1',
 'p2',
 'p3',
 'p4',
 'p5',
 'p6']

In [ ]:
len(_three_of_us), len(_keys)

(19, 19)

In [ ]:
_decm_sols=[key for key in _keys if 'DECM' in final_dict[key].keys()]
_decm_sols==_three_of_us

True

# BIC: DWCM vs. DECM

In [23]:
bic_list=[]
for key in tqdm(_three_of_us):
    _aux_vec=[key]
    for i_m, model in enumerate(['DWCM', 'DECM']):
        bt=final_dict[key][model].sol.best_theta
        _bic=final_dict[key][model].bic(bt)
        _aux_vec.append(_bic)
    bic_list.append(_aux_vec)

  0%|          | 0/19 [00:00<?, ?it/s]

In [24]:
bic_list

[['c0', 6282499.248409398, 8535057.173130982],
 ['c1', 4804761.454930085, 5821905.217401898],
 ['c2', 1990113.4921723416, 2457259.6589885997],
 ['c3', 52080.64383286291, 87744.87442392844],
 ['c4', 164353.57869288858, 278692.1743521256],
 ['e0', 6841219.528808389, 8449338.090186557],
 ['e1', 10098558.555632554, 14554522.445580903],
 ['e2', 3273463.086756979, 3826317.069808797],
 ['e3', 4005915.9737538905, 4769642.732155047],
 ['e4', 550270.2054062487, 861404.4574011486],
 ['e5', 11486.63685663974, 19998.03474565802],
 ['e6', 9862.570943500928, 17345.659417033665],
 ['p0', 2564334.9871918308, 3842625.4264639076],
 ['p1', 597365.3056063815, 980544.097847056],
 ['p2', 1742390.0877645174, 2391140.8608239116],
 ['p3', 136530.85950376553, 227650.35095859392],
 ['p4', 2442950.3872687942, 3232271.7314230995],
 ['p5', 1794611.0690654665, 2906489.4709017407],
 ['p6', 8179.423613893984, 12654.476318440127]]

### Table for $\LaTeX$

In [37]:
bic_list4latex = copy.deepcopy(bic_list)

for i in range(len(bic_list4latex)):
    bic_list4latex[i][0]=name_changer[bic_list[i][0]]
    for j in range(2):
        j_prime=1-j
        if bic_list[i][j+1]<bic_list[i][j_prime+1]:
            bold=True
        else:
            bold=False
        bic_list4latex[i][j+1]=sci_latex(x=bic_list[i][j+1], dec=2, bold=bold)


In [38]:
print(tabulate(bic_list4latex, headers=['DWCM', 'DECM'], tablefmt="latex_raw"))

\begin{tabular}{lll}
\hline
                          & DWCM                         & DECM                \\
\hline
 Crisis - Centrists       & $\mathbf{6.28\times 10^{6}}$ & $8.54\times 10^{6}$ \\
 Crisis - Centre-right    & $\mathbf{4.80\times 10^{6}}$ & $5.82\times 10^{6}$ \\
 Crisis - M5S             & $\mathbf{1.99\times 10^{6}}$ & $2.46\times 10^{6}$ \\
 Crisis - PD              & $\mathbf{5.21\times 10^{4}}$ & $8.77\times 10^{4}$ \\
 Crisis - Media           & $\mathbf{1.64\times 10^{5}}$ & $2.79\times 10^{5}$ \\
 Elections - Centre-right & $\mathbf{6.84\times 10^{6}}$ & $8.45\times 10^{6}$ \\
 Elections - Centre-left  & $\mathbf{1.01\times 10^{7}}$ & $1.46\times 10^{7}$ \\
 Elections - M5S          & $\mathbf{3.27\times 10^{6}}$ & $3.83\times 10^{6}$ \\
 Elections - Centrists    & $\mathbf{4.01\times 10^{6}}$ & $4.77\times 10^{6}$ \\
 Elections - Others       & $\mathbf{5.50\times 10^{5}}$ & $8.61\times 10^{5}$ \\
 Elections - Media        & $\mathbf{1.15\times 10^{4}}$ & $2.0

# BIC: qDECM vs. DECM

In [20]:
_keys=list(final_dict.keys())
_keys.sort()

In [21]:
_three_of_us=[key for key in _keys if len(final_dict[key]) == 3]
_three_of_us

['c0',
 'c1',
 'c2',
 'c3',
 'c4',
 'e0',
 'e1',
 'e2',
 'e3',
 'e4',
 'e5',
 'e6',
 'p0',
 'p1',
 'p2',
 'p3',
 'p4',
 'p5',
 'p6']

In [22]:
len(_three_of_us), len(_keys)

(19, 19)

In [23]:
_decm_sols=[key for key in _keys if 'DECM' in final_dict[key].keys()]
_decm_sols==_three_of_us

True

In [26]:
q_bic_list=[]
for key in tqdm(_three_of_us):
    _aux_vec=[key]
    # qDECM
    bt=final_dict[key]['qDECM'].sol.best_theta
    N=final_dict[key]['qDECM'].N
    _bic=final_dict[key]['qDECM'].bic(bt[:2*N],bt[2*N:])
    _aux_vec.append(_bic)
        
    # DECM
    bt=final_dict[key]['DECM'].sol.best_theta
    _bic=final_dict[key]['DECM'].bic(bt)
    _aux_vec.append(_bic)
    
    q_bic_list.append(_aux_vec)

  0%|          | 0/19 [00:00<?, ?it/s]

In [ ]:
q_bic_list

[['c0', 6282499.248409398, 8535057.173130982],
 ['c1', 4804761.454930085, 5821905.217401898],
 ['c2', 1990113.4921723416, 2457259.6589885997],
 ['c3', 52080.64383286291, 87744.87442392844],
 ['c4', 164353.57869288858, 278692.1743521256],
 ['e0', 6841219.528808389, 8449338.090186557],
 ['e1', 10098558.555632554, 14554522.445580903],
 ['e2', 3273463.086756979, 3826317.069808797],
 ['e3', 4005915.9737538905, 4769642.732155047],
 ['e4', 550270.2054062487, 861404.4574011486],
 ['e5', 11486.63685663974, 19998.03474565802],
 ['e6', 9862.570943500928, 17345.659417033665],
 ['p0', 2564334.9871918308, 3842625.4264639076],
 ['p1', 597365.3056063815, 980544.097847056],
 ['p2', 1742390.0877645174, 2391140.8608239116],
 ['p3', 136530.85950376553, 227650.35095859392],
 ['p4', 2442950.3872687942, 3232271.7314230995],
 ['p5', 1794611.0690654665, 2906489.4709017407],
 ['p6', 8179.423613893984, 12654.476318440127]]

### Table for $\LaTeX$

In [43]:
qbic_list4latex = copy.deepcopy(q_bic_list)

for i in range(len(qbic_list4latex)):
    qbic_list4latex[i][0]=name_changer[q_bic_list[i][0]]
    for j in range(2):
        j_prime=1-j
        if q_bic_list[i][j+1]<q_bic_list[i][j_prime+1]:
            bold=True
        else:
            bold=False
        qbic_list4latex[i][j+1]=sci_latex(x=q_bic_list[i][j+1], dec=4, bold=bold)


In [44]:
print(tabulate(qbic_list4latex, headers=['qDECM', 'DECM'], tablefmt="latex_raw"))

\begin{tabular}{lll}
\hline
                          & qDECM                          & DECM                  \\
\hline
 Crisis - Centrists       & $\mathbf{8.5243\times 10^{6}}$ & $8.5351\times 10^{6}$ \\
 Crisis - Centre-right    & $\mathbf{5.8123\times 10^{6}}$ & $5.8219\times 10^{6}$ \\
 Crisis - M5S             & $\mathbf{2.4533\times 10^{6}}$ & $2.4573\times 10^{6}$ \\
 Crisis - PD              & $\mathbf{8.7698\times 10^{4}}$ & $8.7745\times 10^{4}$ \\
 Crisis - Media           & $\mathbf{2.7849\times 10^{5}}$ & $2.7869\times 10^{5}$ \\
 Elections - Centre-right & $\mathbf{8.4326\times 10^{6}}$ & $8.4493\times 10^{6}$ \\
 Elections - Centre-left  & $\mathbf{1.4535\times 10^{7}}$ & $1.4555\times 10^{7}$ \\
 Elections - M5S          & $\mathbf{3.8186\times 10^{6}}$ & $3.8263\times 10^{6}$ \\
 Elections - Centrists    & $\mathbf{4.7612\times 10^{6}}$ & $4.7696\times 10^{6}$ \\
 Elections - Others       & $\mathbf{8.6089\times 10^{5}}$ & $8.6140\times 10^{5}$ \\
 Elections - Media 

# Log-likelihood: the final showdown

In [47]:
ll_list=[]
for key in tqdm(_three_of_us):
    _aux_vec=[key]
    # DWCM
    bt=final_dict[key]['DWCM'].sol.best_theta
    dwcm_ll=final_dict[key]['DWCM'].neg_log_likelihood(bt)
    _aux_vec.append(dwcm_ll)
    # qDECM
    bt=final_dict[key]['qDECM'].sol.best_theta
    N=final_dict[key]['qDECM'].N
    qdecm_ll=final_dict[key]['qDECM'].neg_log_likelihood_strength(bt[:2*N], bt[2*N:])
    dcm_ll=final_dict[key]['qDECM']._dcm.neg_log_likelihood(bt[:2*N])
    qdecm_ll+=dcm_ll
    _aux_vec.append(qdecm_ll)
    # DECM
    bt=final_dict[key]['DECM'].sol.best_theta
    decm_ll=final_dict[key]['DECM'].neg_log_likelihood(bt)
    _aux_vec.append(decm_ll)


    ll_list.append(_aux_vec)

  0%|          | 0/19 [00:00<?, ?it/s]

In [48]:
ll_list

[['c0', 1849012.6624397433, 1677693.3520048156, 1683054.66303558],
 ['c1', 1741343.9148470503, 1584078.3662517061, 1588878.9834649644],
 ['c2', 703014.796460862, 642584.3186111556, 644545.9302436823],
 ['c3', 7333.638235658133, 6435.676177745893, 6459.069850417579],
 ['c4', 22538.88581111008, 19970.179486091372, 20070.280105394377],
 ['e0', 2331823.087977197, 2038736.7441957027, 2047095.6922392831],
 ['e1', 2569626.222819917, 2308076.1987718, 2317955.112797731],
 ['e2', 1220620.3183958065, 1077056.4089956926, 1080936.0849390326],
 ['e3', 1426013.5037144595, 1226704.7365739672, 1230932.3997525524],
 ['e4', 111044.35464076739, 102265.28775298697, 102520.7325758604],
 ['e5', 1326.8222701488426, 1161.9138936325298, 1166.025056486956],
 ['e6', 1009.5510633141181, 825.5785188716225, 829.3608916441403],
 ['p0', 589920.0661841509, 535231.1786927159, 536817.8584084249],
 ['p1', 93031.64831924315, 78609.85495907255, 78970.03995563282],
 ['p2', 503232.5255966991, 458194.74185143225, 459645.393840

### Table for $\LaTeX$

In [51]:
ll_list4latex = copy.deepcopy(ll_list)

for i in range(len(ll_list4latex)):
    ll_list4latex[i][0]=name_changer[ll_list[i][0]]
    for j in range(3):
        
        if ll_list[i][j+1]==min(ll_list[i][1:]):
            bold=True
        else:
            bold=False
        ll_list4latex[i][j+1]=sci_latex(x=-ll_list[i][j+1], dec=3, bold=bold)


In [52]:
print(tabulate(ll_list4latex, headers=['DWCM', 'qDECM', 'DECM'], tablefmt="latex_raw"))

\begin{tabular}{llll}
\hline
                          & DWCM                  & qDECM                          & DECM                  \\
\hline
 Crisis - Centrists       & $-1.849\times 10^{6}$ & $\mathbf{-1.678\times 10^{6}}$ & $-1.683\times 10^{6}$ \\
 Crisis - Centre-right    & $-1.741\times 10^{6}$ & $\mathbf{-1.584\times 10^{6}}$ & $-1.589\times 10^{6}$ \\
 Crisis - M5S             & $-7.030\times 10^{5}$ & $\mathbf{-6.426\times 10^{5}}$ & $-6.445\times 10^{5}$ \\
 Crisis - PD              & $-7.334\times 10^{3}$ & $\mathbf{-6.436\times 10^{3}}$ & $-6.459\times 10^{3}$ \\
 Crisis - Media           & $-2.254\times 10^{4}$ & $\mathbf{-1.997\times 10^{4}}$ & $-2.007\times 10^{4}$ \\
 Elections - Centre-right & $-2.332\times 10^{6}$ & $\mathbf{-2.039\times 10^{6}}$ & $-2.047\times 10^{6}$ \\
 Elections - Centre-left  & $-2.570\times 10^{6}$ & $\mathbf{-2.308\times 10^{6}}$ & $-2.318\times 10^{6}$ \\
 Elections - M5S          & $-1.221\times 10^{6}$ & $\mathbf{-1.077\times 10^{6}}$ &